In [7]:
# ==============================================================================
# SCRIPT CORRIGIDO V3: REALVISXL LIGHTNING + FIX TENSORFLOW/CIVITAI
# ==============================================================================

import os
import subprocess
import threading
import time

# 1. Vacina contra os erros do Colab (Protobuf, TensorFlow e Wandb)
print("[1/5] Limpando pacotes conflitantes e arrumando dependências...")
!pip uninstall -y tensorflow protobuf wandb > /dev/null 2>&1
!pip install protobuf==3.20.3 > /dev/null 2>&1
!apt-get -y install -qq aria2 > /dev/null 2>&1
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Setup do Forge
print("[2/5] Configurando o WebUI Forge...")
%cd /content
if not os.path.exists('stable-diffusion-webui-forge'):
    !git clone -q https://github.com/lllyasviel/stable-diffusion-webui-forge.git
%cd /content/stable-diffusion-webui-forge

# 3. Baixar Modelo RealVisXL Lightning (Hugging Face)
print("[3/5] Baixando RealVisXL Lightning via Hugging Face...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/SG161222/RealVisXL_V4.0_Lightning/resolve/main/RealVisXL_V4.0_Lightning.safetensors -d /content/stable-diffusion-webui-forge/models/Stable-diffusion -o RealVisXL_V4.0_Lightning.safetensors

# 4. Iniciar Tunnel Cloudflare
print("[4/5] Preparando túnel Cloudflare...")
def start_cloudflared():
    time.sleep(15)
    print("\n" + "="*60)
    print("⏳ GERANDO SEU LINK DO CLOUDFLARE... AGUARDE!")
    print("="*60)

    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        if 'trycloudflare.com' in line:
            url = line.split(" ")[-1].strip()
            print("\n" + "🟢 CLIQUE AQUI PARA ABRIR A INTERFACE: 🟢")
            print(f">>> {url} <<<")
            print("="*60 + "\n")
            break

threading.Thread(target=start_cloudflared, daemon=True).start()

# 5. Launch do Forge
print("[5/5] Iniciando o Forge... (Aguarde o link do Cloudflare aparecer)")
!COMMANDLINE_ARGS="--listen --enable-insecure-extension-access --theme dark --skip-python-version-check" python launch.py

[1/5] Limpando pacotes conflitantes e arrumando dependências...
^C
[2/5] Configurando o WebUI Forge...
/content
/content/stable-diffusion-webui-forge
[3/5] Baixando RealVisXL Lightning via Hugging Face...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e2e421|OK  |       0B/s|/content/stable-diffusion-webui-forge/models/Stable-diffusion/RealVisXL_V4.0_Lightning.safetensors

Status Legend:
(OK):download completed.
[4/5] Preparando túnel Cloudflare...
[5/5] Iniciando o Forge... (Aguarde o link do Cloudflare aparecer)
Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Version: f2.0.1v1.10.1-previous-669-gdfdcbab6
Commit hash: dfdcbab685e57677014f05a3309b48cc87383167
Traceback (most recent call last):
  File "/usr/lib/python3.12/subprocess.py", line 1209, in communicate
    stdout, stderr = self._communicate(input, endtime, timeout)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fi

In [6]:
import os
import time

# 1. Garante que o Cloudflare está instalado (caso o Colab tenha limpado)
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Mata qualquer processo antigo para não dar conflito de porta
!pkill cloudflared

# 3. Roda o Cloudflare no fundo e salva o log
print("⏳ Iniciando o túnel do Cloudflare...")
os.system('cloudflared tunnel --url http://127.0.0.1:7860 > /content/cloudflare.log 2>&1 &')

# 4. Espera 5 segundos para o link ser gerado pelos servidores deles
time.sleep(5)

# 5. Extrai e mostra o link de forma super clara
print("\n" + "="*60)
print("🟢 CLIQUE NO LINK ABAIXO PARA ABRIR A INTERFACE: 🟢")
!grep -o 'https://[^[:space:]]*\.trycloudflare\.com' /content/cloudflare.log
print("="*60 + "\n")

# 6. Inicia o Forge para responder ao link
print("Iniciando o Forge... (A interface vai carregar quando aparecer 'Running on local URL')")
%cd /content/stable-diffusion-webui-forge
!COMMANDLINE_ARGS="--listen --enable-insecure-extension-access --theme dark --skip-python-version-check" python launch.py

⏳ Iniciando o túnel do Cloudflare...

🟢 CLIQUE NO LINK ABAIXO PARA ABRIR A INTERFACE: 🟢
https://married-ethernet-fresh-conclusions.trycloudflare.com

Iniciando o Forge... (A interface vai carregar quando aparecer 'Running on local URL')
/content/stable-diffusion-webui-forge
Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Version: f2.0.1v1.10.1-previous-669-gdfdcbab6
Commit hash: dfdcbab685e57677014f05a3309b48cc87383167
Legacy Preprocessor init warning: Unable to install insightface automatically. Please try run `pip install insightface` manually.
Launching Web UI with arguments: --listen --enable-insecure-extension-access --theme dark --skip-python-version-check
Total VRAM 14913 MB, total RAM 12976 MB
pytorch version: 2.10.0+cu128
Set vram state to: NORMAL_VRAM
Device: cuda:0 Tesla T4 : native
VAE dtype preferences: [torch.float32] -> torch.float32
CUDA Using Stream: False
Using pytorch cross attention
Using pytorch attention for VAE
ControlNet preprocessor location: /content